# 研究与工程思维 2/6：错误分类、切片与辛普森悖论

这不是一节“记术语”的课，而是一节**改变判断过程**的实验课。

| 项目 | 内容 |
|---|---|
| 核心问题 | 总 WER 看起来不错时，系统可能在哪些人和场景上失败？ |
| 迁移价值 | 适用于日志分析、质量看板、用户反馈和任何聚合指标。 |
| 建议投入 | 90～150 分钟；先预测，再运行，再保留被推翻的判断 |
| 通关证据 | 能把结论写成“主张—证据—反证—边界—下一步” |

固定闭环：

```text
观察（发生了什么） → 假设（可能为什么） → 区分性预测 → 最小实验
        ↑                                      ↓
        └──── 更新置信度、记录反例、决定下一步 ────┘
```

**观察不是原因，总分不是解释，相关不是干预效果，运行成功不是结论成立。**


## 课前预测：先暴露自己的判断规则

1. 用一句话回答：总 WER 看起来不错时，系统可能在哪些人和场景上失败？
2. 写出你最可能犯的判断错误，例如“只看平均值”或“看到相关就认定因果”。
3. 为本课写一个可被数据推翻的预测；不要写“应该会更好”这种没有阈值的话。
4. 写出什么结果会让你改变主意。

完成实验后回来修正。保留原答案，因为“怎样改主意”本身就是思维能力证据。


## 一手资料与课程取舍

- [NIST SCTK：按说话人、句子和标签切片报告](https://github.com/usnistgov/SCTK/blob/master/doc/options.htm)
- [Model Cards：按相关条件和群体报告性能](https://research.google/pubs/model-cards-for-model-reporting/)
- [NIST AI RMF：部署场景与分解评估](https://airc.nist.gov/airmf-resources/playbook/measure/)

课程把这些资料转成小型、确定性、可运行的 ASR 实验。示例数据用于理解方法，不代表真实产品结论。


## 1. WER 是入口，不是终点

`WER = (S + D + I) / N`。相同的 10% WER 可能是：专有名词替换、整句幻觉插入、句尾被 VAD 删除，或少数说话人完全不可用。修复手段和风险完全不同。


In [ ]:
def align_words(reference: str, hypothesis: str):
    ref, hyp = reference.split(), hypothesis.split()
    n, m = len(ref), len(hyp)
    dp = [[None] * (m + 1) for _ in range(n + 1)]
    dp[0][0] = (0, 0, 0, 0, [])  # cost, S, D, I, operations
    for i in range(1, n + 1):
        cost, s, d, ins, ops = dp[i-1][0]
        dp[i][0] = (cost+1, s, d+1, ins, ops+[("D", ref[i-1], "*")])
    for j in range(1, m + 1):
        cost, s, d, ins, ops = dp[0][j-1]
        dp[0][j] = (cost+1, s, d, ins+1, ops+[("I", "*", hyp[j-1])])
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            candidates = []
            prev = dp[i-1][j-1]
            if ref[i-1] == hyp[j-1]:
                candidates.append((prev[0], prev[1], prev[2], prev[3], prev[4]+[("C", ref[i-1], hyp[j-1])]))
            else:
                candidates.append((prev[0]+1, prev[1]+1, prev[2], prev[3], prev[4]+[("S", ref[i-1], hyp[j-1])]))
            prev = dp[i-1][j]
            candidates.append((prev[0]+1, prev[1], prev[2]+1, prev[3], prev[4]+[("D", ref[i-1], "*")]))
            prev = dp[i][j-1]
            candidates.append((prev[0]+1, prev[1], prev[2], prev[3]+1, prev[4]+[("I", "*", hyp[j-1])]))
            dp[i][j] = min(candidates, key=lambda item: item[:4])
    return {"N": n, "S": dp[n][m][1], "D": dp[n][m][2], "I": dp[n][m][3], "ops": dp[n][m][4]}

example = align_words("播放 周杰伦 的 稻香", "播放 周杰论 稻香")
print(example)
print("WER=", (example["S"] + example["D"] + example["I"]) / example["N"])
assert (example["S"], example["D"], example["I"]) == (1, 1, 0)


## 2. 同一个总分下面有不同世界

下面的教学记录含设备、噪声、说话人和错误数。先预测哪个切片最差，再计算。小切片必须同时报告分母，不能只报百分比。


In [ ]:
rows = [
    ("s1", "headset", "clean", 12, 0), ("s1", "farfield", "noisy", 10, 4),
    ("s2", "headset", "clean", 11, 1), ("s2", "farfield", "noisy", 9, 3),
    ("s3", "headset", "noisy", 13, 2), ("s3", "farfield", "clean", 10, 2),
    ("s4", "headset", "clean", 8, 0), ("s4", "farfield", "noisy", 12, 5),
]

def grouped_wer(rows, field_index):
    totals = {}
    for row in rows:
        key, words, errors = row[field_index], row[3], row[4]
        totals.setdefault(key, [0, 0])
        totals[key][0] += errors
        totals[key][1] += words
    return {key: {"errors": e, "words": n, "wer": e/n} for key, (e, n) in totals.items()}

print("overall:", sum(r[4] for r in rows) / sum(r[3] for r in rows))
print("device:", grouped_wer(rows, 1))
print("noise:", grouped_wer(rows, 2))
print("speaker:", grouped_wer(rows, 0))


## 3. 辛普森悖论：混合比例能翻转结论

下例中 B 在 clean 和 noisy 内都优于 A，但如果 A 主要在简单样本测试、B 主要在困难样本测试，跨不同测试集的总分会错误地宣称 A 更好。

这不是说“固定同一测试集的模型比较也会凭空翻转”。恰恰相反：它说明**比较必须使用相同样本，或按相同目标分布重新加权**。


In [ ]:
systems = {
    "A": {"clean": (900, 0.05), "noisy": (100, 0.30)},
    "B": {"clean": (100, 0.04), "noisy": (900, 0.25)},
}

def mixed_wer(spec):
    return sum(n * rate for n, rate in spec.values()) / sum(n for n, _ in spec.values())

for name, spec in systems.items():
    print(name, "总WER", f"{mixed_wer(spec):.1%}", "分层", {k: v[1] for k, v in spec.items()})

target_weights = {"clean": 0.6, "noisy": 0.4}
for name, spec in systems.items():
    standardized = sum(target_weights[k] * spec[k][1] for k in target_weights)
    print(name, "按同一部署分布标准化", f"{standardized:.1%}")

assert mixed_wer(systems["A"]) < mixed_wer(systems["B"])
assert all(systems["B"][k][1] < systems["A"][k][1] for k in target_weights)


## 4. 错误频率不等于修复优先级

优先级还取决于影响：把“打开窗帘”识别错通常比把“转账一万元”识别错风险低。可以先用透明的启发式：

`priority = affected_users × error_rate × severity × fixability`

这不是客观真理；它迫使团队公开价值判断，并保留每个因子的依据。


In [ ]:
issues = [
    {"name": "远场删除", "users": 5000, "rate": .18, "severity": 2, "fixability": .7},
    {"name": "金额误识别", "users": 400, "rate": .04, "severity": 10, "fixability": .8},
    {"name": "专名替换", "users": 2400, "rate": .11, "severity": 4, "fixability": .9},
]
for issue in issues:
    issue["priority"] = issue["users"] * issue["rate"] * issue["severity"] * issue["fixability"]
for issue in sorted(issues, key=lambda x: x["priority"], reverse=True):
    print(issue["name"], round(issue["priority"], 1))


## 闭卷挑战

构造一个总体 WER 相同、但 S/D/I 结构和业务风险完全不同的两系统例子。再写一张最小切片表：场景、设备、说话人、长度、SNR、分母、误差率、风险。

回答时强制使用下面的证据卡：

```text
主张：
证据：
最强替代解释：
什么结果会推翻主张：
适用边界：
下一步最小实验：
```


## 最小掌握门禁

- [ ] 我在运行前写了方向和数量级预测。
- [ ] 我能指出示例结论中至少一个替代解释。
- [ ] 我能从空白重写本课核心函数，并用边界输入测试。
- [ ] 我能说明“没有发现差异”和“证明没有差异”的区别。
- [ ] 我把一次被数据推翻的判断写入 `LEARNING_LOG.md`。
- [ ] 我能把本课方法迁移到一个非 ASR 问题。

下一步：第 3 课学习怎样用对照、随机化和因子实验区分原因。
